# Summary_Day6.ipynb  
## 손실함수(Loss) 설계: MSE, MAE, Huber, BCEWithLogits, CrossEntropy, Label Smoothing

이번 6차시는 모델이 얼마나 틀렸는지를 계산하는 **손실함수 Loss Function**를 정리합니다.

핵심 목표:

1. 손실함수의 역할 이해
2. 회귀 손실함수 비교: MSE, MAE, Huber
3. 이상치가 있을 때 손실함수별 민감도 비교
4. 이진 분류에서 MSE와 BCEWithLogitsLoss 비교
5. BCEWithLogitsLoss가 안정적인 이유 이해
6. 다중 분류에서 CrossEntropyLoss 사용법 이해
7. Label Smoothing으로 모델 과신 줄이기
8. Class Weight로 클래스 불균형 대응하기
9. Loss Surface와 안정적 학습 개념 이해

강의 핵심:

```text
손실함수는 오차를 숫자로 측정하고, 모델이 어느 방향으로 학습해야 하는지 알려주는 나침반이다.
```

## 1. 라이브러리 준비

이번 실습에서는 PyTorch, NumPy, Matplotlib, scikit-learn을 사용합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import make_classification, make_regression, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from torch.utils.data import TensorDataset, DataLoader

%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("device:", device)
print("PyTorch:", torch.__version__)

코드 설명:

- `torch`: PyTorch 기본 라이브러리
- `nn`: 손실함수와 신경망 Layer 사용
- `optim`: Optimizer 사용
- `make_classification`: 분류용 예제 데이터 생성
- `make_regression`: 회귀용 예제 데이터 생성
- `load_digits`: 인터넷 없이 사용할 수 있는 손글씨 숫자 데이터
- `device`: GPU가 있으면 cuda, 없으면 cpu 사용

## 2. 손실함수란?

손실함수는 모델의 예측값과 실제 정답 사이의 차이를 숫자로 표현하는 함수입니다.

```text
예측값과 정답이 비슷함 → Loss 작음
예측값과 정답이 다름 → Loss 큼
```

손실함수의 역할:

1. 모델의 오류 정도 측정
2. 학습 방향 결정
3. Optimizer가 따라갈 기준 제공

In [ ]:
y_true_example = torch.tensor([10.0, 20.0, 30.0])
y_pred_example = torch.tensor([12.0, 19.0, 35.0])

error = y_pred_example - y_true_example

print("실제값:", y_true_example)
print("예측값:", y_pred_example)
print("오차:", error)

오차는 `예측값 - 실제값`입니다.

이 오차를 어떤 방식으로 하나의 숫자로 요약하느냐에 따라 MSE, MAE, Huber 같은 손실함수가 됩니다.

## 3. MSE: Mean Squared Error

MSE는 평균 제곱 오차입니다.

공식:

```text
MSE = mean((y_pred - y_true)²)
```

특징:

- 큰 오차에 큰 패널티를 줍니다.
- 미분이 쉬워 학습이 잘 되는 편입니다.
- 이상치에 매우 민감합니다.

In [ ]:
mse_loss = nn.MSELoss()

mse_value = mse_loss(y_pred_example, y_true_example)

print("MSE:", mse_value.item())

# 수동 계산
manual_mse = ((y_pred_example - y_true_example) ** 2).mean()
print("수동 계산 MSE:", manual_mse.item())

코드 설명:

- `nn.MSELoss()`: PyTorch의 MSE 손실함수입니다.
- `** 2`: 오차를 제곱합니다.
- `.mean()`: 제곱 오차의 평균을 계산합니다.

예시 오차가 `[2, -1, 5]`라면 제곱은 `[4, 1, 25]`가 되고 평균은 `10`입니다.

## 4. MAE: Mean Absolute Error

MAE는 평균 절대 오차입니다.

PyTorch에서는 `nn.L1Loss()`를 사용합니다.

공식:

```text
MAE = mean(abs(y_pred - y_true))
```

특징:

- 이상치에 MSE보다 덜 민감합니다.
- 값 해석이 직관적입니다.
- 0 근처에서 미분이 매끄럽지 않을 수 있습니다.

In [ ]:
mae_loss = nn.L1Loss()

mae_value = mae_loss(y_pred_example, y_true_example)

print("MAE:", mae_value.item())

# 수동 계산
manual_mae = torch.abs(y_pred_example - y_true_example).mean()
print("수동 계산 MAE:", manual_mae.item())

코드 설명:

- `nn.L1Loss()`: PyTorch에서 MAE를 계산하는 손실함수입니다.
- `torch.abs()`: 절댓값을 계산합니다.

MAE가 `2.67`이면 평균적으로 약 2.67만큼 틀렸다고 해석할 수 있습니다.

## 5. Huber Loss

Huber Loss는 MSE와 MAE의 장점을 섞은 손실함수입니다.

특징:

- 작은 오차에는 MSE처럼 동작합니다.
- 큰 오차에는 MAE처럼 동작합니다.
- `delta`를 기준으로 두 방식이 바뀝니다.

실전에서 이상치가 있지만 학습 안정성도 필요할 때 유용합니다.

In [ ]:
huber_loss = nn.HuberLoss(delta=1.0)

huber_value = huber_loss(y_pred_example, y_true_example)

print("Huber Loss:", huber_value.item())

`delta=1.0`은 오차가 어느 정도까지는 MSE처럼, 그보다 크면 MAE처럼 보겠다는 기준값입니다.

- `delta` 작음: 큰 오차를 빨리 MAE 방식으로 처리
- `delta` 큼: MSE에 가까운 구간이 넓어짐

## 6. 이상치가 있을 때 손실함수 비교

MSE, MAE, Huber가 이상치에 어떻게 반응하는지 비교합니다.

In [ ]:
y_true_with_outlier = torch.tensor([10.0, 20.0, 30.0, 40.0, 50.0])

y_pred_normal = torch.tensor([11.0, 19.0, 31.0, 39.0, 51.0])
y_pred_with_outlier = torch.tensor([11.0, 19.0, 31.0, 100.0, 51.0])

mse_normal = mse_loss(y_pred_normal, y_true_with_outlier)
mae_normal = mae_loss(y_pred_normal, y_true_with_outlier)
huber_normal = huber_loss(y_pred_normal, y_true_with_outlier)

mse_outlier = mse_loss(y_pred_with_outlier, y_true_with_outlier)
mae_outlier = mae_loss(y_pred_with_outlier, y_true_with_outlier)
huber_outlier = huber_loss(y_pred_with_outlier, y_true_with_outlier)

print("정상 예측")
print("MSE:", mse_normal.item())
print("MAE:", mae_normal.item())
print("Huber:", huber_normal.item())

print("\n이상치 포함 예측")
print("MSE:", mse_outlier.item())
print("MAE:", mae_outlier.item())
print("Huber:", huber_outlier.item())

print("\n증가율")
print("MSE 증가율:", (mse_outlier / mse_normal).item())
print("MAE 증가율:", (mae_outlier / mae_normal).item())
print("Huber 증가율:", (huber_outlier / huber_normal).item())

결과 해석:

- MSE는 오차를 제곱하므로 이상치 하나에도 크게 증가합니다.
- MAE는 절댓값만 보므로 이상치에 비교적 강건합니다.
- Huber는 MSE와 MAE의 중간 성격을 가집니다.

시험 포인트:

```text
이상치가 많으면 MSE보다 MAE 또는 Huber를 고려한다.
```

## 7. 회귀 데이터 생성

이번에는 실제 모델을 MSE, MAE, Huber로 각각 학습해 비교합니다.

일부러 이상치를 추가합니다.

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=10,
    noise=10.0,
    random_state=42
)

n_outliers = int(0.1 * len(y_reg))
outlier_indices = np.random.choice(len(y_reg), n_outliers, replace=False)
y_reg[outlier_indices] += np.random.randn(n_outliers) * 50

X_train, X_test, y_train, y_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train).unsqueeze(1)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test).unsqueeze(1)

print("훈련 데이터:", X_train_t.shape)
print("테스트 데이터:", X_test_t.shape)
print("이상치 개수:", n_outliers)

코드 설명:

- `make_regression()`: 회귀 문제용 데이터를 만듭니다.
- `n_outliers`: 전체 데이터의 10%를 이상치로 만듭니다.
- `StandardScaler`: 입력 feature를 표준화합니다.
- `unsqueeze(1)`: 정답을 `[N]`에서 `[N, 1]` 형태로 바꿉니다.

## 8. 회귀 모델 정의

간단한 MLP 회귀 모델을 만듭니다.

구조:

```text
Linear(10 → 64) → ReLU → Linear(64 → 32) → ReLU → Linear(32 → 1)
```

In [ ]:
class RegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

model_check = RegressionModel()
print(model_check)

코드 설명:

- 입력 feature가 10개이므로 첫 Linear의 입력은 10입니다.
- 마지막 출력은 회귀값 1개이므로 `Linear(32, 1)`입니다.
- ReLU는 비선형성을 추가합니다.

## 9. MSE, MAE, Huber로 각각 학습

같은 모델 구조를 세 가지 손실함수로 따로 학습합니다.

In [ ]:
loss_functions = {
    "MSE": nn.MSELoss(),
    "MAE": nn.L1Loss(),
    "Huber": nn.HuberLoss(delta=1.0)
}

results = {}

for loss_name, criterion in loss_functions.items():
    print(f"\n{loss_name}로 학습 중")

    model = RegressionModel()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    train_losses = []
    num_epochs = 100

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()

        output = model(X_train_t)
        loss = criterion(output, y_train_t)

        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        test_pred = model(X_test_t)

        test_mse = nn.MSELoss()(test_pred, y_test_t).item()
        test_mae = nn.L1Loss()(test_pred, y_test_t).item()
        test_huber = nn.HuberLoss()(test_pred, y_test_t).item()

    results[loss_name] = {
        "train_losses": train_losses,
        "test_mse": test_mse,
        "test_mae": test_mae,
        "test_huber": test_huber,
        "predictions": test_pred
    }

    print("최종 훈련 손실:", train_losses[-1])
    print("테스트 MSE:", test_mse)
    print("테스트 MAE:", test_mae)

학습 루프 설명:

1. `optimizer.zero_grad()`: 이전 gradient 초기화
2. `output = model(X_train_t)`: 예측
3. `loss = criterion(...)`: 손실 계산
4. `loss.backward()`: 역전파
5. `optimizer.step()`: 파라미터 업데이트
6. 테스트 데이터로 성능 평가

## 10. 손실함수별 학습 곡선 비교

세 손실함수의 훈련 손실 변화를 비교합니다.

In [ ]:
for loss_name, result in results.items():
    plt.plot(result["train_losses"], label=loss_name)

plt.xlabel("epoch")
plt.ylabel("training loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

그래프 해석:

- 학습 곡선이 내려가면 손실이 줄어드는 중입니다.
- 손실함수마다 값의 스케일이 다르므로 절대값만 단순 비교하면 안 됩니다.
- 어떤 손실함수를 사용했는지에 따라 학습 안정성과 이상치 민감도가 달라집니다.

## 11. 테스트 MAE 비교

테스트 데이터에서 MAE 기준으로 성능을 비교합니다.

In [ ]:
loss_names = list(results.keys())
test_maes = [results[name]["test_mae"] for name in loss_names]

plt.bar(loss_names, test_maes)
plt.ylabel("Test MAE")
plt.title("Test MAE Comparison")
plt.show()

for name, mae in zip(loss_names, test_maes):
    print(name, "Test MAE:", mae)

MAE는 평균적으로 얼마나 틀렸는지 직관적으로 보여줍니다.

이상치가 섞인 회귀 문제에서는 MSE만 보는 것보다 MAE도 함께 확인하는 것이 좋습니다.

## 12. Huber Loss의 delta 영향

`delta` 값에 따라 Huber Loss의 모양이 달라집니다.

In [ ]:
delta_values = [0.5, 1.0, 2.0, 5.0]
errors = torch.linspace(-10, 10, 200)

for delta in delta_values:
    huber = nn.HuberLoss(delta=delta, reduction="none")
    loss_values = huber(errors, torch.zeros_like(errors))
    plt.plot(errors.numpy(), loss_values.numpy(), label=f"delta={delta}")

mse_values = 0.5 * errors**2
mae_values = torch.abs(errors)

plt.plot(errors.numpy(), mse_values.numpy(), linestyle="--", label="MSE reference")
plt.plot(errors.numpy(), mae_values.numpy(), linestyle=":", label="MAE reference")

plt.xlabel("prediction error")
plt.ylabel("loss value")
plt.title("Huber Loss with Different Delta")
plt.legend()
plt.show()

그래프 해석:

- `delta`가 작으면 큰 오차를 더 빨리 MAE 방식으로 처리합니다.
- `delta`가 크면 MSE처럼 제곱 방식으로 보는 구간이 넓어집니다.
- 일반적으로 `delta=1.0`을 기본값으로 많이 사용합니다.

## 13. 이진 분류: MSE vs BCEWithLogitsLoss

이진 분류는 정답이 0 또는 1인 문제입니다.

예:

```text
스팸 / 정상
질병 / 정상
고양이 / 강아지
```

분류 문제에서는 MSE보다 BCE 계열 손실함수가 더 적합합니다.

In [ ]:
X_cls, y_cls = make_classification(
    n_samples=4000,
    n_features=20,
    n_informative=8,
    weights=[0.6, 0.4],
    random_state=0
)

Xtr, Xte, ytr, yte = train_test_split(
    torch.tensor(X_cls, dtype=torch.float32),
    torch.tensor(y_cls, dtype=torch.float32).unsqueeze(1),
    test_size=0.3,
    random_state=0
)

print("Xtr:", Xtr.shape)
print("ytr:", ytr.shape)
print("Xte:", Xte.shape)
print("yte:", yte.shape)

코드 설명:

- `make_classification()`: 이진 분류용 데이터를 생성합니다.
- `weights=[0.6, 0.4]`: 클래스 비율을 60:40으로 설정합니다.
- `unsqueeze(1)`: 정답 shape을 `[N, 1]`로 맞춥니다.

## 14. 이진 분류 모델 정의

출력은 logit 1개입니다.

주의:

```text
BCEWithLogitsLoss는 sigmoid를 모델 밖에서 따로 적용하지 않습니다.
```

Loss 내부에서 안정적으로 처리합니다.

In [ ]:
class BinNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.m = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.m(x)

binary_model = BinNet()
print(binary_model)

출력값은 확률이 아니라 **raw logit**입니다.

- logit > 0: class 1에 가까움
- logit < 0: class 0에 가까움
- 확률로 볼 때는 `torch.sigmoid(logit)`을 적용합니다.

## 15. 이진 분류 학습 함수

MSE와 BCEWithLogitsLoss를 같은 모델 구조로 비교합니다.

In [ ]:
def run_binary(loss_fn, epochs=8):
    net = BinNet().to(device)
    opt = optim.AdamW(net.parameters(), lr=1e-3)

    Xtr_d = Xtr.to(device)
    ytr_d = ytr.to(device)
    Xte_d = Xte.to(device)
    yte_d = yte.to(device)

    for epoch in range(epochs):
        net.train()
        opt.zero_grad()

        out = net(Xtr_d)
        loss = loss_fn(out, ytr_d)

        loss.backward()
        opt.step()

    net.eval()
    with torch.no_grad():
        logits = net(Xte_d)
        prob = torch.sigmoid(logits)
        pred = (prob > 0.5).float()
        acc = (pred == yte_d).float().mean().item()

    return acc

코드 설명:

- `out = net(Xtr_d)`: raw logit 출력
- MSE 비교 시에는 sigmoid를 적용한 확률과 정답을 비교합니다.
- BCEWithLogitsLoss는 raw logit을 그대로 넣습니다.
- 평가 시에는 sigmoid 후 0.5 기준으로 class를 판단합니다.

## 16. MSE vs BCEWithLogitsLoss 비교

분류 문제에서 두 손실함수를 비교합니다.

In [ ]:
acc_mse = run_binary(
    lambda out, y: nn.MSELoss()(torch.sigmoid(out), y)
)

acc_bce = run_binary(
    nn.BCEWithLogitsLoss()
)

print(f"Binary Accuracy - MSE: {acc_mse:.3f}")
print(f"Binary Accuracy - BCEWithLogits: {acc_bce:.3f}")

결과 해석:

- MSE는 원래 회귀용 손실함수입니다.
- BCEWithLogitsLoss는 이진 분류에 더 적합합니다.
- BCEWithLogitsLoss는 Sigmoid와 BCE를 내부에서 안정적으로 결합합니다.

중요:

```text
BCEWithLogitsLoss를 사용할 때 모델 마지막에 Sigmoid를 붙이지 않는다.
```

## 17. BCE와 BCEWithLogitsLoss 차이

불안정한 방식:

```python
prob = torch.sigmoid(output)
loss = BCELoss(prob, target)
```

안정적인 방식:

```python
loss = BCEWithLogitsLoss(output, target)
```

BCEWithLogitsLoss는 내부에서 수치적으로 안정적인 방식으로 sigmoid + BCE를 처리합니다.

In [ ]:
logits = torch.tensor([[-10.0], [0.0], [10.0]])
targets = torch.tensor([[0.0], [1.0], [1.0]])

bce_logits = nn.BCEWithLogitsLoss()
loss_logits = bce_logits(logits, targets)

prob = torch.sigmoid(logits)
bce = nn.BCELoss()
loss_bce = bce(prob, targets)

print("logits:")
print(logits)

print("\nprobabilities:")
print(prob)

print("\nBCEWithLogitsLoss:", loss_logits.item())
print("Sigmoid + BCELoss:", loss_bce.item())

두 방식의 값은 비슷할 수 있지만, 큰 logit 값에서는 BCEWithLogitsLoss가 더 안전합니다.

실전에서는 이진 분류에 `BCEWithLogitsLoss`를 우선 사용한다고 기억하면 됩니다.

## 18. 다중 분류: CrossEntropyLoss

다중 분류는 3개 이상의 클래스 중 하나를 고르는 문제입니다.

예:

```text
손글씨 숫자 0~9 분류
```

PyTorch의 `CrossEntropyLoss`는 내부에 Softmax가 포함되어 있습니다.

따라서 모델 출력은 확률이 아니라 **raw logits**이어야 합니다.

In [ ]:
sample_logits = torch.tensor([[2.0, 1.0, 0.1]])
sample_target = torch.tensor([0])

ce_loss = nn.CrossEntropyLoss()

sample_loss = ce_loss(sample_logits, sample_target)

print("sample logits:", sample_logits)
print("sample target:", sample_target)
print("CrossEntropyLoss:", sample_loss.item())

입력 형식 주의:

- 예측값: `[batch, class_count]` 형태의 raw logits
- 정답값: `[batch]` 형태의 클래스 인덱스

원-핫 인코딩이 아니라 클래스 번호를 넣습니다.

## 19. 다중 분류 데이터 준비: Digits

원본 실습은 MNIST를 사용하지만, 여기서는 인터넷 없이 실행 가능하도록 scikit-learn의 digits 데이터를 사용합니다.

digits 데이터는 0~9 손글씨 숫자 이미지입니다.

In [ ]:
digits = load_digits()

X_digits = digits.images / 16.0
y_digits = digits.target

X_digits = torch.tensor(X_digits, dtype=torch.float32).unsqueeze(1)
y_digits = torch.tensor(y_digits, dtype=torch.long)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_digits,
    y_digits,
    test_size=0.25,
    random_state=42,
    stratify=y_digits
)

train_loader = DataLoader(
    TensorDataset(X_train_d, y_train_d),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    TensorDataset(X_test_d, y_test_d),
    batch_size=128,
    shuffle=False
)

print("X_train_d:", X_train_d.shape)
print("y_train_d:", y_train_d.shape)
print("X_test_d:", X_test_d.shape)
print("y_test_d:", y_test_d.shape)

데이터 shape 해석:

```text
[N, 1, 8, 8]
```

- `N`: 이미지 개수
- `1`: 흑백 채널
- `8, 8`: 이미지 크기

MNIST는 28x28이지만, digits는 8x8입니다.

## 20. 작은 CNN 모델 정의

손글씨 숫자를 분류하는 작은 CNN을 만듭니다.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(32 * 2 * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.net(x)

cnn_check = SmallCNN()
print(cnn_check)

코드 설명:

- `Conv2d`: 이미지에서 특징을 추출합니다.
- `ReLU`: 비선형성을 추가합니다.
- `MaxPool2d`: 이미지 크기를 줄입니다.
- `Flatten`: 2D feature map을 1차원 벡터로 펼칩니다.
- 마지막 `Linear(64, 10)`: 10개 숫자 클래스에 대한 logit을 출력합니다.

## 21. 다중 분류 학습/평가 함수

손실함수를 바꿔가며 모델을 학습하고 정확도를 계산합니다.

In [ ]:
def train_eval_multiclass(criterion, epochs=5):
    model = SmallCNN().to(device)
    opt = optim.AdamW(model.parameters(), lr=2e-3)

    for epoch in range(epochs):
        model.train()

        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            opt.zero_grad()

            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            loss.backward()
            opt.step()

    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_x)
            pred = logits.argmax(dim=1)

            correct += (pred == batch_y).sum().item()
            total += batch_y.size(0)

    return correct / total

코드 흐름:

1. `logits = model(batch_x)`: raw logits 출력
2. `loss = criterion(logits, batch_y)`: CrossEntropy 계열 손실 계산
3. `loss.backward()`: gradient 계산
4. `opt.step()`: 파라미터 업데이트
5. `argmax(dim=1)`: 가장 큰 logit의 class를 예측값으로 선택

## 22. Label Smoothing

Label Smoothing은 정답을 100% 확신하지 않도록 부드럽게 만드는 기법입니다.

예: 클래스가 3개이고 정답이 0일 때

```text
일반 정답: [1, 0, 0]
Label Smoothing: [0.933, 0.033, 0.033]
```

효과:

- 모델 과신 완화
- 일반화 성능 향상
- 과적합 방지

In [ ]:
crit_ls = nn.CrossEntropyLoss(label_smoothing=0.1)

acc_ls = train_eval_multiclass(crit_ls, epochs=5)

print(f"Accuracy with Label Smoothing: {acc_ls:.3f}")

`label_smoothing=0.1`은 정답 확률 일부를 다른 클래스에 나눠주는 설정입니다.

모델이 정답 클래스에만 과도하게 확신하는 것을 줄입니다.

## 23. Class Weight 적용

클래스 불균형이 있을 때 특정 클래스에 더 큰 가중치를 줄 수 있습니다.

In [ ]:
weights = torch.tensor(
    [1, 1, 1, 1, 1, 1.2, 1, 1.2, 1, 1.2],
    dtype=torch.float32
).to(device)

crit_weighted = nn.CrossEntropyLoss(weight=weights)

acc_weighted = train_eval_multiclass(crit_weighted, epochs=5)

print(f"Accuracy with Weighted CE: {acc_weighted:.3f}")

코드 설명:

- `weight`: 클래스별 손실 가중치입니다.
- 특정 클래스의 가중치를 높이면 그 클래스 오답에 더 큰 패널티를 줍니다.
- 의료 데이터처럼 소수 클래스가 중요한 경우 사용할 수 있습니다.

## 24. 클래스 불균형 대응 전략

강의에서는 클래스 불균형 문제를 매우 중요하게 다뤘습니다.

예:

```text
정상 99%
질병 1%
```

이때 모델이 모두 정상이라고 예측해도 정확도는 99%입니다.

하지만 질병을 하나도 못 찾는 쓸모없는 모델이 됩니다.

In [ ]:
imbalance_strategies = {
    "Weighted Cross Entropy": "소수 클래스의 손실 가중치를 높임",
    "Over-sampling": "소수 클래스 데이터를 늘림",
    "Under-sampling": "다수 클래스 데이터를 줄임",
    "Focal Loss": "어려운 샘플에 더 집중함",
    "SMOTE": "소수 클래스의 합성 데이터를 만듦"
}

for key, value in imbalance_strategies.items():
    print(f"{key}: {value}")

전략 선택 기준:

- 약한 불균형: Weighted CE
- 데이터가 충분하면 Over-sampling
- 극심한 불균형: Focal Loss 고려
- 작은 데이터셋: SMOTE 고려 가능

## 25. Focal Loss 개념

Focal Loss는 쉬운 샘플의 영향은 줄이고, 어려운 샘플에 더 집중하게 만드는 손실함수입니다.

객체 탐지나 segmentation처럼 불균형이 심한 문제에서 자주 사용됩니다.

대표 설정:

```text
gamma = 2
alpha = 0.25
```

In [ ]:
def focal_loss_binary(logits, targets, alpha=0.25, gamma=2.0):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    )

    prob = torch.sigmoid(logits)
    pt = torch.where(targets == 1, prob, 1 - prob)

    focal_weight = alpha * (1 - pt) ** gamma

    loss = focal_weight * bce

    return loss.mean()

sample_logits = torch.tensor([[2.0], [0.0], [-2.0]])
sample_targets = torch.tensor([[1.0], [1.0], [0.0]])

print("Focal Loss:", focal_loss_binary(sample_logits, sample_targets).item())

코드 설명:

- `BCEWithLogits`를 기본 손실로 사용합니다.
- `pt`: 모델이 정답에 부여한 확률입니다.
- `(1 - pt) ** gamma`: 쉬운 샘플의 손실을 줄이고 어려운 샘플의 손실을 상대적으로 키웁니다.

## 26. Loss Surface 개념

Loss Surface는 가중치 공간에서 손실값이 어떻게 변하는지 나타낸 지형입니다.

비유:

```text
가중치 = 위치
Loss = 고도
학습 = 낮은 계곡을 찾아 내려가기
```

좋은 Loss Surface는 부드럽고 넓은 최소값을 가지는 경우입니다.

In [ ]:
loss_surface_tips = [
    "BatchNorm 사용: 입력 분포 안정화",
    "Skip Connection 사용: 깊은 네트워크 학습 안정화",
    "He/Xavier 초기화: 적절한 시작점 제공",
    "작은 batch size: 넓은 최소값 탐색에 도움",
    "Gradient Clipping: gradient 폭발 방지",
    "Warm-up: 초반 학습률을 천천히 증가",
    "Loss 모니터링: 이상 징후 조기 발견"
]

for tip in loss_surface_tips:
    print("-", tip)

시험 포인트:

손실함수만 잘 고르는 것이 전부가 아닙니다.

좋은 학습을 위해서는 다음도 함께 중요합니다.

- 초기화
- BatchNorm
- 학습률
- Scheduler
- Gradient Clipping
- 데이터 불균형 대응

## 27. 주요 함수 / 변수 / 약어 정리

| 이름 | 의미 | 설명 |
|---|---|---|
| `Loss` | 손실 | 모델이 틀린 정도 |
| `Cost Function` | 비용 함수 | 손실함수와 거의 같은 의미로 사용 |
| `MSE` | Mean Squared Error | 평균 제곱 오차 |
| `MAE` | Mean Absolute Error | 평균 절대 오차 |
| `L1Loss` | MAE | PyTorch의 MAE 손실 |
| `Huber` | Huber Loss | MSE와 MAE 절충 |
| `delta` | Huber 경계값 | MSE/MAE 전환 기준 |
| `BCE` | Binary Cross Entropy | 이진 분류 손실 |
| `BCEWithLogitsLoss` | 안정적 BCE | Sigmoid + BCE를 내부에서 안정적으로 처리 |
| `logit` | raw score | Sigmoid/Softmax 전 원시 출력 |
| `CrossEntropyLoss` | 다중 분류 손실 | Softmax가 내부 포함됨 |
| `Label Smoothing` | 라벨 스무딩 | 정답 확률을 부드럽게 만듦 |
| `Class Weight` | 클래스 가중치 | 소수 클래스 손실을 더 크게 반영 |
| `Focal Loss` | 어려운 샘플 집중 손실 | 불균형 분류에 사용 |
| `alpha` | Focal 가중치 | 클래스 불균형 보정 |
| `gamma` | Focal 집중도 | 쉬운 샘플 손실 감소 정도 |
| `Loss Surface` | 손실 곡면 | 가중치 공간의 손실 지형 |

## 28. 시험용 요약

```text
손실함수 = 모델이 얼마나 틀렸는지 숫자로 표현하고 학습 방향을 정하는 기준
```

핵심 정리:

- MSE는 오차를 제곱하므로 큰 오차와 이상치에 민감합니다.
- MAE는 절댓값을 사용하므로 이상치에 상대적으로 강건합니다.
- Huber Loss는 작은 오차에는 MSE처럼, 큰 오차에는 MAE처럼 동작합니다.
- 이진 분류에는 MSE보다 BCE 계열 손실이 적합합니다.
- `BCEWithLogitsLoss`는 Sigmoid와 BCE를 내부에서 안정적으로 처리합니다.
- `BCEWithLogitsLoss`를 사용할 때 모델 마지막에 Sigmoid를 붙이지 않습니다.
- 다중 분류에는 `CrossEntropyLoss`를 사용합니다.
- `CrossEntropyLoss`에는 Softmax가 내부적으로 포함되어 있습니다.
- `CrossEntropyLoss`의 정답은 one-hot이 아니라 클래스 인덱스입니다.
- Label Smoothing은 모델의 과신을 줄이고 일반화를 돕습니다.
- 클래스 불균형이 심하면 accuracy만 보면 안 됩니다.
- Weighted CE는 소수 클래스 손실에 더 큰 가중치를 줍니다.
- Focal Loss는 어려운 샘플에 더 집중하게 만듭니다.
- Loss Surface는 가중치 공간에서 손실값의 지형입니다.
- 안정적 학습에는 손실함수뿐 아니라 초기화, BatchNorm, 학습률, gradient clipping도 중요합니다.